# 用神经网络的思路理解逻辑回归

欢迎你第一次（必填）编程作业！你将构建一个逻辑回归分类器来识别猫。 作业将引导你了解如何使用神经网络思维来做到这一点，并且还将磨练你对深度学习的直觉。

**说明：**
- 不要在代码中使用循环（for/while），除非说明明确要求你这样做。
- 使用 `np.dot(X,Y)` 计算点积。

**你将学会：**
- 构建学习算法的总体架构，包括：
    - 初始化参数
    - 计算代价函数及其梯度
    - 使用优化算法（梯度下降） 
- 按正确的顺序将上述所有三个函数收集到主模型函数中。

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/neural-networks-deep-learning/supplement/iLwon/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录
- [1 - 依赖库](#1)
- [2 - 问题概述](#2)
    - [练习 1](#ex-1)
    - [练习 2](#ex-2)
- [3 - 学习算法的总体架构](#3)
- [4 - 构建算法的各个部分](#4)
    - [4.1 - 辅助函数](#4-1)
        - [练习 3 - sigmoid](#ex-3)
    - [4.2 - 初始化参数](#4-2)
        - [练习 4 - initialize_with_zeros](#ex-4)
    - [4.3 - 前向传播与反向传播](#4-3)
        - [练习 5 - propagate](#ex-5)
    - [4.4 - 优化](#4-4)
        - [练习 6 - optimize](#ex-6)
        - [练习 7 - predict](#ex-7)
- [5 - 将所有函数合并到模型中](#5)
    - [练习 8 - model](#ex-8)
- [6 - 进一步分析（可选/不计分练习）](#6)
- [7 - 使用你自己的图像进行测试（可选/不计分的练习）](#7)

<a name='1'></a>
## 1 - 依赖库 ##

首先，让我们运行下面的单元来导入在此作业期间需要的所有包。 
- [NumPy](https://numpy.org/doc/1.20/) 是使用 Python 进行科学计算的基础包。
- [h5py](http://www.h5py.org) 是一个与存储在 H5 文件中的数据集交互的通用包。
- [matplotlib](http://matplotlib.org) 是一个著名的 Python 绘图库。
- [太平](https://pillow.readthedocs.io/en/stable/) 和 [scipy](https://www.scipy.org/) 在这里用来测试你的模型，最后用你自己的图片来测试。

In [ ]:
### v1.2

In [ ]:
import numpy as np
import copy
import matplotlib.pyplot as plt
import h5py
import scipy
from PIL import Image
from scipy import ndimage
from lr_utils import load_dataset
from public_tests import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

<a name='2'></a>
## 2 - 问题概述 ##

**问题陈述**：给你一个数据集（“data.h5”），其中包含：
    - 训练集的 m_train 图像标记为猫 (y=1) 或非猫 (y=0)
    - 测试集的 m_test 图像标记为猫或非猫
    - 每个图像的形状为 (num_px, num_px, 3)，其中 3 表示 3 个通道 (RGB)。因此，每个图像都是正方形（高度 = num_px）和（宽度 = num_px）。

你将构建一个简单的图像识别算法，可以正确地将图片分类为猫或非猫。

让我们更熟悉一下数据集。通过运行以下代码加载数据。

In [ ]:
# 加载数据(cat/non-cat)
train_set_x_orig, train_set_y, test_set_x_orig, test_set_y, classes = load_dataset()

我们在图像数据集（训练和测试）的末尾添加了“_orig”，因为我们要对它们进行预处理。预处理后，我们最终将得到 train_set_x 和 test_set_x（标签 train_set_y 和 test_set_y 不需要任何预处理）。

train_set_x_orig 和 test_set_x_orig 的每一行都是一个代表图像的数组。你可以通过运行以下代码来可视化示例。你也可以随意更改 `index` 值并重新运行以查看其他图像。 

In [ ]:
# 图片示例
index = 25
plt.imshow(train_set_x_orig[index])
print ("y = " + str(train_set_y[:, index]) + ", it's a '" + classes[np.squeeze(train_set_y[:, index])].decode("utf-8") +  "' picture.")

深度学习中的许多软件错误都来自于矩阵/向量维度不合适。如果你能保持矩阵/向量维度正确，你将在消除许多错误方面大有帮助。 

<a name='ex-1'></a>
### 练习 1
查找以下值：
    - m_train（训练样本的数量）
    - m_test（测试示例数）
    - num_px（= 训练图像的高度 = 宽度）
请记住，`train_set_x_orig` 是一个 NumPy 数组，形状为（m_train、num_px、num_px、3）。例如，你可以通过写入 `train_set_x_orig.shape[0]` 来访问 `m_train`。

In [ ]:
#（约3行代码）
# m_train = 
# m_test = 
# num_px = 
# YOUR CODE STARTS HERE


# YOUR CODE ENDS HERE

print ("Number of training examples: m_train = " + str(m_train))
print ("Number of testing examples: m_test = " + str(m_test))
print ("Height/Width of each image: num_px = " + str(num_px))
print ("Each image is of size: (" + str(num_px) + ", " + str(num_px) + ", 3)")
print ("train_set_x shape: " + str(train_set_x_orig.shape))
print ("train_set_y shape: " + str(train_set_y.shape))
print ("test_set_x shape: " + str(test_set_x_orig.shape))
print ("test_set_y shape: " + str(test_set_y.shape))

**m_train、m_test 和 num_px 的预期输出**： 
<table style="width:15%">
  <tr>
    <td> m_train </td>
    <td> 209 </td> 
  </tr>
  
  <tr>
    <td>m_test</td>
    <td> 50 </td> 
  </tr>
  
  <tr>
    <td>num_px</td>
    <td> 64 </td> 
  </tr>
  
</table>


为了方便起见，你现在应该在形状 (num_px $*$ num_px $*$ 3, 1) 的 NumPy 数组中重塑形状 (num_px, num_px, 3) 的图像。之后，我们的训练（和测试）数据集是一个 NumPy 数组，其中每列代表一个扁平图像。应有 m_train（分别为 m_test）列。

<a name='ex-2'></a>
### 练习 2
重塑训练和测试数据集，以便将大小 (num_px, num_px, 3) 的图像展平为形状 (num\_px $*$ num\_px $*$ 3, 1) 的单个向量。

当你想要将形状为 (a,b,c,d) 的矩阵 X 展平为形状为 $(b \times c \times d, a)$ 的矩阵 X_flatten 时，一个技巧是使用： 
```python
X_flatten = X.reshape(X.shape[0], -1).T      # X.T 是 X 的转置
```

In [ ]:
# 重塑训练和测试示例
#（约2行代码）
# train_set_x_flatten = ...
# test_set_x_flatten = ...
# YOUR CODE STARTS HERE


# YOUR CODE ENDS HERE

# 检查第二张图像的前 10 个像素是否位于正确的位置
assert np.alltrue(train_set_x_flatten[0:10, 1] == [196, 192, 190, 193, 186, 182, 188, 179, 174, 213]), "Wrong solution. Use (X.shape[0], -1).T."
assert np.alltrue(test_set_x_flatten[0:10, 1] == [115, 110, 111, 137, 129, 129, 155, 146, 145, 159]), "Wrong solution. Use (X.shape[0], -1).T."

print ("train_set_x_flatten shape: " + str(train_set_x_flatten.shape))
print ("train_set_y shape: " + str(train_set_y.shape))
print ("test_set_x_flatten shape: " + str(test_set_x_flatten.shape))
print ("test_set_y shape: " + str(test_set_y.shape))

**预期输出**： 

<table style="width:35%">
  <tr>
    <td>train_set_x_flatten 形状</td>
    <td> (12288, 209)</td> 
  </tr>
  <tr>
    <td>train_set_y 形状</td>
    <td>(1, 209)</td> 
  </tr>
  <tr>
    <td>test_set_x_flatten 形状</td>
    <td>(12288, 50)</td> 
  </tr>
  <tr>
    <td>test_set_y 形状</td>
    <td>(1, 50)</td> 
  </tr>
</table>

为了表示彩色图像，必须为每个像素指定红、绿、蓝通道（RGB），因此像素值实际上是由 0 到 255 之间的三个数字组成的向量。

机器学习中的一个常见预处理步骤是对数据集进行中心化和标准化，这意味着你从每个示例中减去整个 NumPy 数组的平均值，然后将每个示例除以整个 NumPy 数组的标准差。但对于图片数据集，更简单、更方便，只需将数据集的每一行除以 255（像素通道的最大值）几乎同样有效。

<!-- 训练模型时，先对输入进行加权求和并加上偏置，得到神经元的激活值，再通过反向传播计算梯度并训练模型。各个特征的取值范围应尽量相近，以降低梯度爆炸的风险。后续课程会进一步介绍这一点。 !-->

让我们标准化我们的数据集。

In [ ]:
train_set_x = train_set_x_flatten / 255.
test_set_x = test_set_x_flatten / 255.

<div style="color:blue;">

**你需要记住的是：**

预处理新数据集的常见步骤是：
- 找出问题的维度和形状（m_train、m_test、num_px，...）
- 重塑数据集，使每个示例现在都是大小为 (num_px \* num_px \* 3, 1) 的向量
- “标准化”数据

</div>

<a name='3'></a>
## 3 - 学习算法的总体架构 ##

是时候设计一个简单的算法来区分猫图像和非猫图像了。

你将使用神经网络思维构建逻辑回归。下图解释了为什么**逻辑回归实际上是一个非常简单的神经网络！**

<img src="images/LogReg_kiank.png" style="width:650px;height:400px;">

**算法的数学表达式**：

例如 $x^{(i)}$：

$$
z^{(i)} = w^T x^{(i)} + b \tag{1}
$$

$$
\hat{y}^{(i)} = a^{(i)} = sigmoid(z^{(i)})\tag{2}
$$

$$
\mathcal{L}(a^{(i)}, y^{(i)}) =  - y^{(i)}  \log(a^{(i)}) - (1-y^{(i)} )  \log(1-a^{(i)})\tag{3}
$$



然后通过对所有训练样本求和来计算代价：

$$
J = \frac{1}{m} \sum_{i=1}^m \mathcal{L}(a^{(i)}, y^{(i)})\tag{6}
$$

**关键步骤**：
在本练习中，你将执行以下步骤： 
    - 初始化模型参数
    - 通过最小化代价来学习模型的参数  
    - 使用学习到的参数进行预测（在测试集上）
    - 分析结果并得出结论

<a name='4'></a>
## 4 - 构建算法的各个部分 ## 

构建神经网络的主要步骤是：
1. 定义模型结构（例如输入特征的数量） 
2. 初始化模型参数
3. 循环：
    - 计算电流损耗（前向传播）
    - 计算当前梯度 (反向传播)
    - 更新参数（梯度下降）

你经常单独构建 1-3 个并将它们集成到一个函数中，我们称之为 `model()`。

<a name='4-1'></a>
### 4.1 - 辅助函数

<a name='ex-3'></a>
### 练习 3 - sigmoid
使用“Python 基础知识”中的代码，实现 `sigmoid()`。正如你在上图中看到的，你需要计算 $sigmoid(z) = \frac{1}{1 + e^{-z}}$ 来为 $z = w^T x + b$ 进行预测。使用 np.exp()。

In [ ]:
# GRADED FUNCTION: sigmoid

def sigmoid(z):
    """
    计算 z 的 sigmoid

    参数：
    z -- 任意大小的标量或 NumPy 数组。

    返回：
    s -- sigmoid(z)
    """

    #（≈ 1 行代码）
    # s = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return s

In [ ]:
print ("sigmoid([0, 2]) = " + str(sigmoid(np.array([0,2]))))

sigmoid_test(sigmoid)

In [ ]:
x = np.array([0.5, 0, 2.0])
output = sigmoid(x)
print(output)

<a name='4-2'></a>
### 4.2 - 初始化参数

<a name='ex-4'></a>
### 练习 4 - initialize_with_zeros
在下面的单元格中实现参数初始化。你必须将 w 初始化为零向量。如果你不知道要使用什么 NumPy 函数，请在 NumPy 库的文档中查找 np.zeros()。

In [ ]:
# GRADED FUNCTION: initialize_with_zeros

def initialize_with_zeros(dim):
    """
    这个函数为 w 创建一个形状为 (dim, 1) 的零向量，并将 b 初始化为 0。
    
    论证：
    dim -- 我们想要的 w 向量的大小（或本例中的参数数量）
    
    返回：
    w -- 形状为 (dim, 1) 的初始化向量
    b -- float 类型的初始化标量（对应于偏置）
    """
    
    # （约2行代码）
    # w = ...
    # b = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE

    return w, b

In [ ]:
dim = 2
w, b = initialize_with_zeros(dim)

assert type(b) == float
print ("w = " + str(w))
print ("b = " + str(b))

initialize_with_zeros_test_1(initialize_with_zeros)
initialize_with_zeros_test_2(initialize_with_zeros)


<a name='4-3'></a>
### 4.3 - 前向传播与反向传播

现在参数已初始化，你可以执行“前向”和“后向”传播步骤来学习参数。

<a name='ex-5'></a>
### 练习 5 - propagate
实现一个函数 `propagate()` 来计算代价函数及其梯度。

**提示**：

前向传播:
- 你得到X
- 你计算$A = \sigma(w^T X + b) = (a^{(1)}, a^{(2)}, ..., a^{(m-1)}, a^{(m)})$
- 你计算出代价函数：$J = -\frac{1}{m}\sum_{i=1}^{m}(y^{(i)}\log(a^{(i)})+(1-y^{(i)})\log(1-a^{(i)}))$

以下是你将使用的两个公式：

$$
\frac{\partial J}{\partial w} = \frac{1}{m}X(A-Y)^T\tag{7}
$$

$$
\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^m (a^{(i)}-y^{(i)})\tag{8}
$$

In [ ]:
# GRADED FUNCTION: propagate

def propagate(w, b, X, Y):
    """
    实现代价函数及其梯度以进行上述传播

    参数：
    w -- 权重，大小为 (num_px * num_px * 3, 1) 的 NumPy 数组
    b -- 偏置，标量
    X -- 大小数据（num_px * num_px * 3，示例数量）
    Y -- 真实的 "label" 向量（如果非猫则包含 0，如果猫则包含 1），大小为（1，示例数）

    返回：
    grads -- 包含权重和偏置梯度的字典
            (dw -- 损失相对于 w 的梯度，形状与 w 相同)
            (db -- 损失相对于 b 的梯度，形状与 b 相同)
    cost -- 逻辑回归的负对数似然代价
    
    Tips:
    - 逐步编写传播代码。 np.log()、np.dot()
    """
    
    m = X.shape[1]
    
    # 前向传播（从 X 到代价）
    #（约2行代码）
    # 计算激活值
    # A = ...
    # 使用 np.dot 执行乘法来计算代价。 
    # 并且不要使用循环求和。
    # cost = ...                                
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE

    # 反向传播（计算梯度）
    #（约2行代码）
    # dw = ...
    # db = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    cost = np.squeeze(np.array(cost))

    
    grads = {"dw": dw,
             "db": db}
    
    return grads, cost

In [ ]:
w =  np.array([[1.], [2]])
b = 1.5

# X 使用 3 个示例，每个示例有 2 个特征
# 每个示例均按列堆叠
X = np.array([[1., -2., -1.], [3., 0.5, -3.2]])
Y = np.array([[1, 1, 0]])
grads, cost = propagate(w, b, X, Y)

assert type(grads["dw"]) == np.ndarray
assert grads["dw"].shape == (2, 1)
assert type(grads["db"]) == np.float64


print ("dw = " + str(grads["dw"]))
print ("db = " + str(grads["db"]))
print ("cost = " + str(cost))

propagate_test(propagate)

**预期输出**

```
dw = [[ 0.25071532]
 [-0.06604096]]
db = -0.1250040450043965
cost = 0.15900537707692405
```

<a name='4-4'></a>
### 4.4 - 优化
- 你已初始化参数。
- 你还可以计算代价函数及其梯度。
- 现在，你想使用梯度下降更新参数。

<a name='ex-6'></a>
### 练习 6 - optimize
记下优化函数。目标是通过最小化代价函数 $J$ 来学习 $w$ 和 $b$。对于参数$\theta$，更新规则为$ \theta = \theta - \alpha \text{ } d\theta$，其中$\alpha$为学习率。

In [ ]:
# GRADED FUNCTION: optimize

def optimize(w, b, X, Y, num_iterations=100, learning_rate=0.009, print_cost=False):
    """
    这个函数通过运行梯度下降算法来优化 w 和 b
    
    参数：
    w -- 权重，大小为 (num_px * num_px * 3, 1) 的 NumPy 数组
    b -- 偏置，标量
    X -- 形状数据（num_px * num_px * 3，示例数）
    Y -- true "label" 向量（如果非猫则包含 0，如果猫则包含 1），形状（1，示例数）
    num_iterations -- 优化循环的迭代次数
    learning_rate -- 梯度下降更新规则的学习率
    print_cost -- True 打印每100步的损失
    
    返回：
    params -- 包含权重 w 和偏置 b 的字典
    grads -- 包含代价函数相对于权重和偏置的梯度的字典
    costs -- 优化期间计算的所有代价的列表，这将用于绘制学习曲线。
    
    Tips:
    你基本上需要写下两个步骤并迭代它们：
        1) 计算当前参数的代价和梯度。使用 propagate()。
        2) 使用梯度下降规则更新 w 和 b 的参数。
    """
    
    w = copy.deepcopy(w)
    b = copy.deepcopy(b)
    
    costs = []
    
    for i in range(num_iterations):
        # （约1行代码）
        # 代价及梯度计算 
        # grads, cost = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
        # 从 grads 字典中取出导数
        dw = grads["dw"]
        db = grads["db"]
        
        # 更新规则（约2行代码）
        # w = ...
        # b = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
        # 记录代价
        if i % 100 == 0:
            costs.append(cost)
        
            # 打印每 100 次训练迭代的代价
            if print_cost:
                print ("Cost after iteration %i: %f" %(i, cost))
    
    params = {"w": w,
              "b": b}
    
    grads = {"dw": dw,
             "db": db}
    
    return params, grads, costs

In [ ]:
params, grads, costs = optimize(w, b, X, Y, num_iterations=100, learning_rate=0.009, print_cost=False)

print ("w = " + str(params["w"]))
print ("b = " + str(params["b"]))
print ("dw = " + str(grads["dw"]))
print ("db = " + str(grads["db"]))
print("Costs = " + str(costs))

optimize_test(optimize)

<a name='ex-7'></a>
### 练习 7 - predict
前面的函数将输出学习到的w和b。我们能够使用 w 和 b 来预测数据集 X 的标签。实现 `predict()` 函数。计算预测有两个步骤：

1. 计算$\hat{Y} = A = \sigma(w^T X + b)$

2. 将 a 的条目转换为 0（如果激活值 <= 0.5）或 1（如果激活值 > 0.5），将预测存储在向量 `Y_prediction` 中。如果你愿意，可以在 `for` 循环中使用 `if`/`else` 语句（尽管也有一种方法可以实现向量化）。 

In [ ]:
# GRADED FUNCTION: predict

def predict(w, b, X):
    '''
    使用学习到的逻辑回归参数 (w, b) 预测标签是 0 还是 1
    
    参数：
    w -- 权重，大小为 (num_px * num_px * 3, 1) 的 NumPy 数组
    b -- 偏置，标量
    X -- 大小数据（num_px * num_px * 3，示例数量）
    
    返回：
    Y_prediction -- 一个 NumPy 数组（向量），包含 X 中示例的所有预测 (0/1)
    '''
    
    m = X.shape[1]
    Y_prediction = np.zeros((1, m))
    w = w.reshape(X.shape[0], 1)
    
    # 计算向量 "A" 预测图片中出现猫的概率
    #（≈ 1 行代码）
    # A = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    for i in range(A.shape[1]):
        
        # 将概率 A[0,i] 转换为实际预测 p[0,i]
        #（约4行代码）
        # if A[0, i] > ____ :
        #     Y_prediction[0,i] = 
        # else:
        #     Y_prediction[0,i] = 
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
    
    return Y_prediction

In [ ]:
w = np.array([[0.1124579], [0.23106775]])
b = -0.3
X = np.array([[1., -1.1, -3.2],[1.2, 2., 0.1]])
print ("predictions = " + str(predict(w, b, X)))

predict_test(predict)

<div style="color:blue;">

**要记住什么：**
    
你已经实现了多个函数：
- 初始化 (w,b)
- 迭代优化损失以学习参数 (w,b)：
    - 计算代价及其梯度 
    - 使用梯度下降更新参数
- 使用学习到的 (w,b) 来预测给定示例集的标签

</div>

<a name='5'></a>
## 5 - 将所有函数合并到模型中 ##

现在，你将看到如何通过按正确的顺序将所有构建块（前面部分中实现的函数）组合在一起来构建整个模型。

<a name='ex-8'></a>
### 练习 8 - model
实现模型函数。使用以下符号：
    - Y_prediction_test 你对测试集的预测
    - Y_prediction_train 用于你对训练集的预测
    - optimize() 输出的参数、梯度、代价

In [ ]:
# GRADED FUNCTION: model

def model(X_train, Y_train, X_test, Y_test, num_iterations=2000, learning_rate=0.5, print_cost=False):
    """
    通过调用你之前实现的函数构建逻辑回归模型
    
    参数：
    X_train -- 训练集由形状为 (num_px * num_px * 3, m_train) 的 NumPy 数组表示
    Y_train -- 由形状为 (1, m_train) 的 NumPy 数组（向量）表示的训练标签
    X_test -- 测试集由形状为 (num_px * num_px * 3, m_test) 的 NumPy 数组表示
    Y_test -- 由形状为 (1, m_test) 的 NumPy 数组（向量）表示的测试标签
    num_iterations -- 表示优化参数的迭代次数的超参数
    learning_rate -- 代表 optimize() 更新规则中使用的学习率的超参数
    print_cost -- 设置为 True 以打印每 100 次迭代的代价
    
    返回：
    d -- 包含模型信息的字典。
    """
    # （≈ 1 行代码）   
    # 用零初始化参数
    # 并使用 "shape" 函数获得 X_train 的第一个维度
    # w, b = ...
    
    #（≈ 1 行代码）
    # 梯度下降 
    # params, grads, costs = ...
    
    # 从字典 "params" 中检索参数 w 和 b
    # w = ...
    # b = ...
    
    # 预测测试/训练集示例（≈ 2 行代码）
    # Y_prediction_test = ...
    # Y_prediction_train = ...
    
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE

    # 打印训练/测试错误
    if print_cost:
        print("train accuracy: {} %".format(100 - np.mean(np.abs(Y_prediction_train - Y_train)) * 100))
        print("test accuracy: {} %".format(100 - np.mean(np.abs(Y_prediction_test - Y_test)) * 100))

    
    d = {"costs": costs,
         "Y_prediction_test": Y_prediction_test, 
         "Y_prediction_train" : Y_prediction_train, 
         "w" : w, 
         "b" : b,
         "learning_rate" : learning_rate,
         "num_iterations": num_iterations}
    
    return d

In [ ]:
from public_tests import *

model_test(model)

如果你通过了所有测试，请运行以下单元来训练你的模型。

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=2000, learning_rate=0.005, print_cost=True)

**评论**：训练准确率接近100%。这是一个很好的健全性检查：你的模型正在工作并且具有足够高的容量来适合训练数据。测试准确率为70%。考虑到我们使用的小数据集并且逻辑回归是一个线性分类器，对于这个简单的模型来说实际上还不错。但不用担心，下周你将构建一个更好的分类器！

此外，你会发现模型明显过拟合训练数据。稍后在本专项课程中，你将学习如何减少过拟合，例如使用正则化。使用下面的代码（并更改 `index` 变量），你可以查看对测试集图片的预测。

In [ ]:
# 错误分类的图片示例。
index = 1
plt.imshow(test_set_x[:, index].reshape((num_px, num_px, 3)))
print ("y = " + str(test_set_y[0,index]) + ", you predicted that it is a \"" + classes[int(logistic_regression_model['Y_prediction_test'][0,index])].decode("utf-8") +  "\" picture.")

我们还绘制代价函数和梯度。

In [ ]:
# 绘制学习曲线（含代价）
costs = np.squeeze(logistic_regression_model['costs'])
plt.plot(costs)
plt.ylabel('cost')
plt.xlabel('iterations (per hundreds)')
plt.title("Learning rate =" + str(logistic_regression_model["learning_rate"]))
plt.show()

**解释**：
你可以看到代价在下降。表明正在学习参数。但是，你会发现你可以在训练集上进一步训练模型。尝试增加上面单元格中的迭代次数并重新运行单元格。你可能会看到训练集精度上升，但测试集精度下降。这称为过拟合。 

<a name='6'></a>
## 6 - 进一步分析（可选/不计分练习） ##

恭喜你构建了你的第一个图像分类模型。让我们进一步分析，并检查学习率 $\alpha$ 的可能选择。 

#### 选择学习率 ####

**提醒**：
为了使梯度下降正常工作，你必须明智地选择学习率。 学习率 $\alpha$ 决定我们更新参数的速度。如果学习率太大，我们可能会“超出”最佳值。同样，如果它太小，我们将需要太多迭代才能收敛到最佳值。这就是为什么使用经过良好调校的学习率至关重要。

让我们将模型的学习曲线与几种学习率选择进行比较。运行下面的单元格。这大约需要 1 分钟。也可以随意尝试与我们初始化 `learning_rates` 变量包含的三个值不同的值，看看会发生什么。 

In [ ]:
learning_rates = [0.01, 0.001, 0.0001]
models = {}

for lr in learning_rates:
    print ("Training a model with learning rate: " + str(lr))
    models[str(lr)] = model(train_set_x, train_set_y, test_set_x, test_set_y, num_iterations=1500, learning_rate=lr, print_cost=False)
    print ('\n' + "-------------------------------------------------------" + '\n')

for lr in learning_rates:
    plt.plot(np.squeeze(models[str(lr)]["costs"]), label=str(models[str(lr)]["learning_rate"]))

plt.ylabel('cost')
plt.xlabel('iterations (hundreds)')

legend = plt.legend(loc='upper center', shadow=True)
frame = legend.get_frame()
frame.set_facecolor('0.90')
plt.show()

**解释**： 
- 不同的学习率会产生不同的代价，从而产生不同的预测结果。
- 如果学习率过大（例如 0.01），代价可能上下振荡，甚至发散。不过，在本例中，学习率为 0.01 时最终仍能取得较低的代价。
- 较低的代价并不意味着更好的模型。你必须检查是否可能存在过拟合。当训练准确率远高于测试准确率时，就会发生这种情况。
- 在深度学习中，我们通常建议你： 
    - 选择能更好地最小化代价函数的学习率。
    - 如果你的模型过拟合，请使用其他技术来减少过拟合。 （我们将在后面的视频中讨论这一点。） 


<a name='7'></a>
## 7 - 使用你自己的图像进行测试（可选/不计分的练习） ##

恭喜你完成了这个作业。你可以使用自己的图像并查看模型的输出。为此：
    1. 单击此 Notebook 上部栏中的“文件”，然后单击“打开”以转到你的 Coursera Hub。
    2. 将图像添加到 Jupyter Notebook 目录的“images”文件夹中
    3. 在以下代码中更改图像的名称
    4. 运行代码并检查算法是否正确（1 = 猫，0 = 非猫）！

In [ ]:
# 将其更改为图像文件的名称
my_image = "my_image.jpg"   

# 我们对图像进行预处理以适合你的算法。
fname = "images/" + my_image
image = np.array(Image.open(fname).resize((num_px, num_px)))
plt.imshow(image)
image = image / 255.
image = image.reshape((1, num_px * num_px * 3)).T
my_predicted_image = predict(logistic_regression_model["w"], logistic_regression_model["b"], image)

print("y = " + str(np.squeeze(my_predicted_image)) + ", your algorithm predicts a \"" + classes[int(np.squeeze(my_predicted_image)),].decode("utf-8") +  "\" picture.")

<div style="color:blue;">

**从这个作业中要记住什么：**
1. 预处理数据集很重要。
2. 你分别实现了每个函数：initialize()、propagate()、optimize()。然后你构建了 model()。
3. 调整学习率（这是“超参数”的一个示例）可以对算法产生很大的影响。你将在本课程后面看到更多这样的示例！

</div>

最后，如果你愿意，我们邀请你在这款 Notebook 上尝试不同的东西。请确保在尝试任何操作之前提交。提交后，你可以使用的内容包括：
    - 玩转学习率和迭代次数
    - 尝试不同的初始化方法并比较结果
    - 测试其他预处理（将数据居中，或将每行除以其标准差）

参考书目：
- http://www.wildml.com/2015/09/implementing-a-neural-network-from-scratch/
- https://stats.stackexchange.com/questions/211436/why-do-we-normalize-images-by-subtracting-the-datasets-image-mean-and-not-the-c